# ESPN API: exploración de datos NFL

La API de ESPN no es oficial ni está documentada. No necesita API key, pero puede cambiar sin aviso, así que todo el código usa acceso defensivo (`dig`, `.get`) y cachea las respuestas en disco.

**Hosts que vamos a usar**

| Host | Para qué sirve |
|---|---|
| `site.api.espn.com/apis/site/v2` | Scoreboard, equipos, rosters, resumen de partido (box score, drives, jugadas) |
| `site.api.espn.com/apis/v2` | Standings completas (la ruta `site/v2/.../standings` devuelve casi nada) |
| `site.web.api.espn.com/apis/common/v3` | Datos por atleta: gamelog y stats |
| `sports.core.api.espn.com/v2` | Odds, probabilidades y detalle fino (respuestas con links `$ref`) |

**Contenido**
1. Setup y helpers
2. Scoreboard de la semana
3. Equipos y roster
4. Standings
5. Detalle de un partido (box score, scoring plays, drives)
6. Temporada completa histórica
7. EDA rápido: ventaja de local, márgenes, totales
8. Gamelog de un jugador
9. Odds desde el core API
10. Ideas para modelar

In [ ]:
import json
import time
import hashlib
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

SITE = "https://site.api.espn.com/apis/site/v2/sports"          # scoreboard, teams, roster, summary
SITE_V2 = "https://site.api.espn.com/apis/v2/sports"            # standings completas
WEB = "https://site.web.api.espn.com/apis/common/v3/sports"     # atletas: gamelog, stats
CORE = "https://sports.core.api.espn.com/v2/sports"             # odds, probabilidades, detalle fino

CACHE_DIR = Path("espn_cache")
CACHE_DIR.mkdir(exist_ok=True)

session = requests.Session()


def get_json(url, params=None, use_cache=True, max_age_hours=6, retries=3, pause=0.3):
    """GET con cache en disco y reintentos.

    La API no es oficial: no hay key ni limites publicados, pero conviene
    cachear y no disparar cientos de requests seguidos.
    """
    params = params or {}
    key = hashlib.md5((url + json.dumps(params, sort_keys=True)).encode()).hexdigest()
    path = CACHE_DIR / f"{key}.json"

    if use_cache and path.exists():
        age_h = (time.time() - path.stat().st_mtime) / 3600
        if age_h < max_age_hours:
            return json.loads(path.read_text())

    last_err = None
    for attempt in range(retries):
        try:
            r = session.get(url, params=params, timeout=20)
            if r.status_code == 404:
                print(f"404: {r.url}")
                return {}
            r.raise_for_status()
            data = r.json()
            path.write_text(json.dumps(data))
            time.sleep(pause)
            return data
        except Exception as e:  # red, 5xx, json invalido
            last_err = e
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f"Fallo {url} {params}: {last_err}")


def peek(obj, depth=2, indent=0, max_items=12):
    """Imprime el 'esqueleto' de un JSON: llaves, tipos y largos. Util para explorar."""
    pad = "  " * indent
    if isinstance(obj, dict):
        for i, (k, v) in enumerate(obj.items()):
            if i >= max_items:
                print(f"{pad}... (+{len(obj) - max_items} llaves)")
                break
            kind = type(v).__name__
            extra = f" [{len(v)}]" if isinstance(v, (list, dict)) else f" = {str(v)[:40]!r}"
            print(f"{pad}{k}: {kind}{extra}")
            if depth > 0 and isinstance(v, (dict, list)) and v:
                peek(v, depth - 1, indent + 1, max_items)
    elif isinstance(obj, list) and obj:
        print(f"{pad}[0] de {len(obj)}:")
        peek(obj[0], depth, indent + 1, max_items)


def dig(obj, *path, default=None):
    """Acceso seguro a llaves anidadas: dig(ev, 'competitions', 0, 'venue', 'fullName')."""
    cur = obj
    for p in path:
        try:
            cur = cur[p]
        except (KeyError, IndexError, TypeError):
            return default
    return cur


def to_num(x):
    """Convierte '24' o '3/10' o '--' a numero cuando se puede."""
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan


def parse_standings(data):
    """Aplana standings: recorre children (conferencias, divisiones, grupos) de forma recursiva."""
    rows = []

    def walk(node, group=None):
        name = node.get("name") or node.get("abbreviation") or group
        entries = dig(node, "standings", "entries", default=[]) or []
        for e in entries:
            row = {
                "group": name,
                "team_id": dig(e, "team", "id"),
                "team": dig(e, "team", "displayName"),
                "abbr": dig(e, "team", "abbreviation"),
            }
            for s in e.get("stats", []):
                col = s.get("name") or s.get("type")
                if not col:
                    continue
                val = s.get("value")
                row[col] = val if val is not None else s.get("displayValue")
            rows.append(row)
        for child in node.get("children", []) or []:
            walk(child, name)

    walk(data)
    return pd.DataFrame(rows)

In [ ]:
# Configuracion
SPORT, LEAGUE = "football", "nfl"
HIST_SEASON = 2025      # temporada completa para analisis historico
CURRENT_SEASON = 2026   # temporada en curso
MY_TEAM = "MIA"         # cambia por cualquier abreviatura: KC, DAL, PHI...

BASE = f"{SITE}/{SPORT}/{LEAGUE}"
print(BASE)

## 1. Scoreboard

Es el endpoint más útil para empezar. Cada elemento de `events` es un partido; dentro, `competitions[0].competitors` trae los dos equipos con `homeAway`, `score`, `records` y a veces `leaders`.

Parámetros clave: `dates` (YYYYMMDD, YYYYMM o el año), `seasontype` (1 pretemporada, 2 regular, 3 playoffs) y `week`. Sin parámetros devuelve la semana actual.

Primero miramos el esqueleto del JSON con `peek` y después lo aplanamos a un DataFrame.

In [ ]:
sb = get_json(f"{BASE}/scoreboard")
peek(sb, depth=1)
print("\nEstructura de un evento:")
if sb.get("events"):
    peek(sb["events"][0], depth=2)

In [ ]:
def parse_scoreboard(data):
    rows = []
    for ev in data.get("events", []):
        comp = dig(ev, "competitions", 0, default={})
        teams = {c.get("homeAway"): c for c in comp.get("competitors", [])}
        home, away = teams.get("home", {}), teams.get("away", {})

        def record(c):
            for r in c.get("records", []) or []:
                if r.get("type") in ("total", "overall") or r.get("name") == "overall":
                    return r.get("summary")
            return dig(c, "records", 0, "summary")

        odds = dig(comp, "odds", 0, default={}) or {}
        rows.append({
            "event_id": ev.get("id"),
            "date": ev.get("date"),
            "season": dig(ev, "season", "year"),
            "season_type": dig(ev, "season", "type"),
            "week": dig(ev, "week", "number"),
            "name": ev.get("shortName"),
            "status": dig(comp, "status", "type", "description") or dig(ev, "status", "type", "description"),
            "completed": dig(comp, "status", "type", "completed", default=False),
            "home": dig(home, "team", "abbreviation"),
            "away": dig(away, "team", "abbreviation"),
            "home_score": to_num(home.get("score")),
            "away_score": to_num(away.get("score")),
            "home_record": record(home),
            "away_record": record(away),
            "venue": dig(comp, "venue", "fullName"),
            "neutral_site": comp.get("neutralSite"),
            "attendance": comp.get("attendance"),
            "odds_details": odds.get("details"),
            "over_under": odds.get("overUnder"),
            "spread": odds.get("spread"),
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], utc=True)
    return df


games_now = parse_scoreboard(sb)
games_now

## 2. Equipos y roster

`/teams` trae los 32 equipos anidados en `sports[0].leagues[0].teams`. Con el `id` del equipo pedimos `/teams/{id}/roster`.

En la NFL el roster viene agrupado por unidad (`offense`, `defense`, `specialTeam`), cada grupo con su lista `items`. En otras ligas viene plano, por eso el parser maneja ambos casos.

In [ ]:
teams_raw = get_json(f"{BASE}/teams")
teams = pd.DataFrame([
    {
        "team_id": t["team"].get("id"),
        "abbr": t["team"].get("abbreviation"),
        "name": t["team"].get("displayName"),
        "location": t["team"].get("location"),
        "color": t["team"].get("color"),
        "logo": dig(t, "team", "logos", 0, "href"),
    }
    for t in dig(teams_raw, "sports", 0, "leagues", 0, "teams", default=[])
])
teams

In [ ]:
def parse_roster(data):
    rows = []
    groups = data.get("athletes", [])
    # NFL: lista de grupos con 'items'. Otras ligas: lista plana de atletas.
    if groups and isinstance(groups[0], dict) and "items" in groups[0]:
        flat = [(g.get("position"), a) for g in groups for a in g.get("items", [])]
    else:
        flat = [(None, a) for a in groups]
    for unit, a in flat:
        rows.append({
            "athlete_id": a.get("id"),
            "name": a.get("fullName") or a.get("displayName"),
            "unit": unit,
            "pos": dig(a, "position", "abbreviation"),
            "jersey": a.get("jersey"),
            "age": a.get("age"),
            "height_in": a.get("height"),
            "weight_lb": a.get("weight"),
            "exp_years": dig(a, "experience", "years"),
            "college": dig(a, "college", "name"),
            "status": dig(a, "status", "name"),
        })
    return pd.DataFrame(rows)


team_id = teams.loc[teams["abbr"] == MY_TEAM, "team_id"].iloc[0]
roster = parse_roster(get_json(f"{BASE}/teams/{team_id}/roster"))
print(roster["unit"].value_counts(dropna=False))
roster.head(15)

## 3. Standings

Ojo con la ruta: las standings completas salen de `/apis/v2/...` y no de `/apis/site/v2/...`. La respuesta viene anidada en `children` (conferencias, y a veces divisiones adentro). Cada equipo trae una lista `stats` con `name` y `value`; el helper `parse_standings` la convierte en columnas.

In [ ]:
st_raw = get_json(f"{SITE_V2}/{SPORT}/{LEAGUE}/standings", params={"season": CURRENT_SEASON})
standings = parse_standings(st_raw)
print("Columnas disponibles:", list(standings.columns))
cols = [c for c in ["group", "abbr", "wins", "losses", "ties", "winPercent",
                    "pointsFor", "pointsAgainst", "differential", "streak", "playoffSeed"]
        if c in standings.columns]
standings[cols].sort_values(["group", "winPercent"] if "winPercent" in cols else ["group"],
                            ascending=[True, False] if "winPercent" in cols else True)

## 4. Detalle de un partido: `summary`

`/summary?event={id}` es el endpoint más rico. Llaves principales:

* `boxscore.teams`: estadísticas de equipo (yardas, primeros downs, posesión, turnovers)
* `boxscore.players`: stat lines por jugador, agrupadas por categoría (passing, rushing, receiving, defensive...)
* `scoringPlays`: cada anotación con el marcador acumulado
* `drives.previous`: cada serie ofensiva con resultado, yardas y duración
* `winprobability`, `pickcenter`, `injuries`, `gameInfo`

Tomamos un partido terminado de la semana actual (o de la temporada histórica si no hay).

In [ ]:
done = games_now[games_now["completed"]] if not games_now.empty else games_now
if done.empty:
    done = parse_scoreboard(get_json(f"{BASE}/scoreboard",
                                     params={"dates": HIST_SEASON, "seasontype": 2, "week": 1}))
EVENT_ID = done["event_id"].iloc[0]
print("Partido:", done["name"].iloc[0], "| event_id:", EVENT_ID)

summ = get_json(f"{BASE}/summary", params={"event": EVENT_ID})
peek(summ, depth=0)

In [ ]:
def team_stats(summary):
    rows = []
    for t in dig(summary, "boxscore", "teams", default=[]):
        row = {"team": dig(t, "team", "abbreviation")}
        for s in t.get("statistics", []):
            row[s.get("name") or s.get("label")] = s.get("displayValue")
        rows.append(row)
    return pd.DataFrame(rows).set_index("team").T if rows else pd.DataFrame()


def player_stats(summary):
    # Una fila por jugador y categoria (passing, rushing, receiving...)
    out = {}
    for t in dig(summary, "boxscore", "players", default=[]):
        abbr = dig(t, "team", "abbreviation")
        for cat in t.get("statistics", []):
            labels = cat.get("labels") or cat.get("keys") or []
            rows = []
            for a in cat.get("athletes", []):
                row = {"team": abbr,
                       "athlete_id": dig(a, "athlete", "id"),
                       "player": dig(a, "athlete", "displayName")}
                row.update(dict(zip(labels, a.get("stats", []))))
                rows.append(row)
            out.setdefault(cat.get("name"), []).extend(rows)
    return {k: pd.DataFrame(v) for k, v in out.items()}


team_stats(summ)

In [ ]:
ps = player_stats(summ)
print("Categorias:", list(ps))
for cat in ["passing", "rushing", "receiving"]:
    if cat in ps:
        print(f"\n{cat.upper()}")
        display(ps[cat])

In [ ]:
scoring = pd.DataFrame([{
    "q": dig(p, "period", "number"),
    "clock": dig(p, "clock", "displayValue"),
    "team": dig(p, "team", "abbreviation"),
    "type": dig(p, "type", "text"),
    "text": p.get("text"),
    "away": p.get("awayScore"),
    "home": p.get("homeScore"),
} for p in summ.get("scoringPlays", [])])

drives = pd.DataFrame([{
    "team": dig(d, "team", "abbreviation"),
    "result": d.get("result") or d.get("displayResult"),
    "plays": d.get("offensivePlays"),
    "yards": d.get("yards"),
    "time": dig(d, "timeElapsed", "displayValue"),
    "start_q": dig(d, "start", "period", "number"),
    "start_yardline": dig(d, "start", "yardLine"),
    "desc": d.get("description"),
} for d in dig(summ, "drives", "previous", default=[])])

display(scoring)
display(drives.head(12))
if not drives.empty:
    print(drives.groupby(["team", "result"]).size().unstack(fill_value=0))

## 5. Temporada completa

Para modelar necesitamos historia. Recorremos las 18 semanas de temporada regular (`seasontype=2`) y las 5 rondas de playoffs (`seasontype=3`) de `HIST_SEASON`. Son unos 23 requests, cacheados en disco, así que la segunda corrida es instantánea.

El resultado se guarda en CSV para reutilizarlo en otros notebooks.

In [ ]:
def load_season(season, reg_weeks=18, post_weeks=5):
    frames = []
    for stype, weeks in [(2, reg_weeks), (3, post_weeks)]:
        for wk in range(1, weeks + 1):
            data = get_json(f"{BASE}/scoreboard",
                            params={"dates": season, "seasontype": stype, "week": wk},
                            max_age_hours=24 * 30)
            frames.append(parse_scoreboard(data))
    df = pd.concat([f for f in frames if not f.empty], ignore_index=True)
    return df.drop_duplicates("event_id")


season = load_season(HIST_SEASON)
season = season[season["completed"]].copy()
season["margin"] = season["home_score"] - season["away_score"]
season["total"] = season["home_score"] + season["away_score"]
season["home_win"] = (season["margin"] > 0).astype(int)
season.to_csv(f"nfl_games_{HIST_SEASON}.csv", index=False)
print(season.shape)
season.head()

## 6. EDA rápido

Tres preguntas básicas que cualquier modelo de NFL tiene que respetar:

1. **Ventaja de local:** históricamente vale entre 1.5 y 2.5 puntos y ha venido bajando.
2. **Números clave:** los márgenes se concentran en 3 y 7 por la forma de anotar (field goal y touchdown). Esto importa mucho para spreads.
3. **Totales:** la distribución de puntos totales por partido, base para over/under.

In [ ]:
reg = season[(season["season_type"] == 2) & (season["neutral_site"] != True)]
print(f"Partidos: {len(reg)}")
print(f"Local gana: {reg['home_win'].mean():.1%}")
print(f"Margen medio del local: {reg['margin'].mean():+.2f} pts")
print(f"Total medio: {reg['total'].mean():.1f} pts")

abs_margin = reg["margin"].abs().value_counts().sort_index()
print("\nMargenes mas frecuentes:")
print(abs_margin.sort_values(ascending=False).head(8))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
abs_margin.loc[:30].plot.bar(ax=axes[0], color="#4C72B0")
axes[0].set_title("Margen absoluto de victoria")
axes[0].set_xlabel("puntos")
reg["total"].plot.hist(bins=25, ax=axes[1], color="#55A868")
axes[1].set_title("Puntos totales por partido")
plt.tight_layout()
plt.show()

In [ ]:
# Tabla por equipo: puntos a favor, en contra y diferencial
home = reg.rename(columns={"home": "team", "home_score": "pf", "away_score": "pa"})[["team", "pf", "pa"]]
away = reg.rename(columns={"away": "team", "away_score": "pf", "home_score": "pa"})[["team", "pf", "pa"]]
by_team = (pd.concat([home, away])
           .groupby("team")
           .agg(games=("pf", "size"), pf=("pf", "mean"), pa=("pa", "mean")))
by_team["diff"] = by_team["pf"] - by_team["pa"]
by_team.sort_values("diff", ascending=False).round(1)

## 7. Gamelog de un jugador

El host `site.web.api.espn.com/apis/common/v3` trae datos por atleta. El gamelog combina tres piezas:

* `names` / `labels`: el nombre de cada columna de stats
* `events`: metadata por partido (rival, fecha, resultado)
* `seasonTypes[].categories[].events`: los valores de stats por partido

Aquí tomamos el QB titular del roster de `MY_TEAM`. Esto es lo que después se cruzaría con precios de cartas.

In [ ]:
def parse_gamelog(data):
    names = data.get("names") or data.get("labels") or []
    meta = data.get("events", {}) or {}
    rows = []
    for st in data.get("seasonTypes", []) or []:
        for cat in st.get("categories", []) or []:
            for e in cat.get("events", []) or []:
                eid = e.get("eventId")
                m = meta.get(eid, {})
                row = {
                    "season_type": st.get("displayName"),
                    "event_id": eid,
                    "date": m.get("gameDate"),
                    "week": m.get("week"),
                    "opp": dig(m, "opponent", "abbreviation"),
                    "at_vs": m.get("atVs"),
                    "result": m.get("gameResult"),
                    "score": m.get("score"),
                }
                row.update(dict(zip(names, e.get("stats", []))))
                rows.append(row)
    df = pd.DataFrame(rows)
    if not df.empty and "date" in df:
        df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
        df = df.sort_values("date")
    return df


qbs = roster[roster["pos"] == "QB"]
ATHLETE_ID = qbs["athlete_id"].iloc[0] if not qbs.empty else roster["athlete_id"].iloc[0]
print("Jugador:", roster.loc[roster["athlete_id"] == ATHLETE_ID, "name"].iloc[0], ATHLETE_ID)

gl_raw = get_json(f"{WEB}/{SPORT}/{LEAGUE}/athletes/{ATHLETE_ID}/gamelog",
                  params={"season": HIST_SEASON})
gamelog = parse_gamelog(gl_raw)
if gamelog.empty:
    print("Gamelog vacio; esqueleto de la respuesta:")
    peek(gl_raw, depth=1)
gamelog

## 8. Odds desde el core API

El core API responde con links `$ref` en lugar de datos completos. Para odds de un partido:

`/football/leagues/nfl/events/{id}/competitions/{id}/odds`

En NFL el id de la competición es igual al del evento. Para partidos ya jugados muchas veces viene vacío; para partidos futuros suele traer spread, total y moneylines por proveedor.

In [ ]:
upcoming = games_now[~games_now["completed"]] if not games_now.empty else games_now
ODDS_EVENT = upcoming["event_id"].iloc[0] if not upcoming.empty else EVENT_ID

odds_raw = get_json(f"{CORE}/{SPORT}/leagues/{LEAGUE}/events/{ODDS_EVENT}"
                    f"/competitions/{ODDS_EVENT}/odds")
odds = pd.DataFrame([{
    "provider": dig(o, "provider", "name"),
    "details": o.get("details"),
    "spread": o.get("spread"),
    "over_under": o.get("overUnder"),
    "home_ml": dig(o, "homeTeamOdds", "moneyLine"),
    "away_ml": dig(o, "awayTeamOdds", "moneyLine"),
} for o in odds_raw.get("items", [])])
odds if not odds.empty else print("Sin odds para este evento")

## 9. Ideas para modelar

* **Elo o rating de poder** usando `season`: es el baseline estándar y se calibra con la ventaja de local medida arriba.
* **Features por partido** desde `summary`: yardas por jugada, turnovers, eficiencia en tercera oportunidad, resultado de drives.
* **Comparar contra el mercado:** el modelo solo vale si le gana al spread de cierre, no si acierta ganadores.
* **Puente con las cartas:** el gamelog por jugador es la variable explicativa natural para precios de cartas (yardas, touchdowns, partidos jugados, lesiones).